# Fase 3 - Camada Gold, Análise e Perguntas de negócio.
## 1 Configurando
### 1.1 Importando bibliotecas

In [8]:

import pandas as pd
import matplotlib.pyplot as plt
import banco

### 1.2 Conectando ao banco de dados

In [9]:
conexao = banco.conectar()

print("Conexão realizada com sucesso!")

Conexão realizada com sucesso!


### 1.3 Funções complementares

In [ ]:
def consultar(sql):
    return pd.read_sql(sql, conexao)


def reais(valor):
    texto = f'{valor:,.2f}'
    return 'R$ ' + texto.replace(',', 'X').replace('.', ',').replace('X', '.')


# Configurações do pandas para exibir valores monetários em reais

pd.options.display.float_format = lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

### 1.4 Teste

In [11]:
print("Conectado ao MySQL com sucesso.")

Conectado ao MySQL com sucesso.


## 2 Explorando e validando as tabelas

In [ ]:
sql = """
SELECT 'silver_viagem' AS tabela, COUNT(*) AS total_registros
FROM silver_viagem

UNION ALL

SELECT 'silver_trecho', COUNT(*)
FROM silver_trecho

UNION ALL

SELECT 'silver_passagem', COUNT(*)
FROM silver_passagem

UNION ALL

SELECT 'silver_pagamento', COUNT(*)
FROM silver_pagamento;
"""

df = consultar(sql)
df

## 3 Criação da camada Gold e view

In [13]:
cursor = conexao.cursor()

cursor.execute("DROP VIEW IF EXISTS vw_gold_viagens")
cursor.execute("DROP TABLE IF EXISTS gold_viagens")

cursor.execute("""
CREATE TABLE gold_viagens AS

SELECT
    v.id_viagem,
    v.num_proposta,
    v.nome_orgao_superior,
    v.destinos,
    v.data_inicio,
    v.data_fim,
    v.duracao_dias,
    v.valor_total,

    pg.nome_orgao_pagador,
    pg.nome_ug_pagadora,
    pg.tipo_pagamento,
    pg.valor_total_pagamentos,

    tr.quantidade_trechos,
    tr.meio_transporte,
    tr.destino_uf

FROM silver_viagem v

LEFT JOIN (
    SELECT
        id_viagem,
        MAX(nome_orgao_pagador) AS nome_orgao_pagador,
        MAX(nome_ug_pagadora) AS nome_ug_pagadora,
        MAX(tipo_pagamento) AS tipo_pagamento,
        SUM(valor) AS valor_total_pagamentos
    FROM silver_pagamento
    GROUP BY id_viagem
) pg
ON v.id_viagem = pg.id_viagem

LEFT JOIN (
    SELECT
        id_viagem,
        COUNT(*) AS quantidade_trechos,
        MAX(meio_transporte) AS meio_transporte,
        MAX(destino_uf) AS destino_uf
    FROM silver_trecho
    GROUP BY id_viagem
) tr
ON v.id_viagem = tr.id_viagem
""")

cursor.execute("""
CREATE VIEW vw_gold_viagens AS
SELECT *
FROM gold_viagens
""")

conexao.commit()

print("Camada GOLD criada com sucesso!")

Camada GOLD criada com sucesso!


In [ ]:
sql = """
SELECT *
FROM gold_viagens
LIMIT 10;
"""

df = consultar(sql)

df

## 4 Perguntas de Negócio

### Primeira pergunta de negócio - Os 5 órgãos com maior custo total

Nesta análise, será identificado quais órgãos apresentaram o maior custo total com viagens, considerando a soma dos pagamentos registrados.

In [ ]:
sql = """
SELECT
    nome_orgao_pagador,
    SUM(valor) AS custo_total
FROM silver_pagamento
GROUP BY nome_orgao_pagador
ORDER BY custo_total DESC
LIMIT 5;
"""

df = consultar(sql)

df

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(df["nome_orgao_pagador"], df["custo_total"])
plt.title("Os 5 órgãos com maior custo total")
plt.xlabel("Custo Total (R$)")
plt.ylabel("Órgão")
plt.tight_layout()
plt.show()

#### Conclusão

Os resultados mostram quais órgãos concentraram os maiores gastos com viagens no período analisado. Essa informação pode auxiliar na identificação dos órgãos com maior volume de despesas e servir de apoio para análises de planejamento orçamentário e controle de gastos públicos.

### Segunda pergunta de negócio - Os 3 destinos com maior custo médio por viagem?

Esta análise identifica os destinos que apresentaram o maior custo médio por viagem, considerando apenas destinos com pelo menos 30 viagens registradas para evitar distorções estatísticas.

In [ ]:
sql = """
SELECT
    destinos,
    AVG(valor_total) AS custo_medio
FROM silver_viagem
WHERE destinos <> ''
GROUP BY destinos
HAVING COUNT(*) >= 30
ORDER BY custo_medio DESC
LIMIT 3;
"""

df = consultar(sql)

df

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(df["destinos"], df["custo_medio"])
plt.title("Os 3 destinos com maior custo médio")
plt.xlabel("Destino")
plt.ylabel("Custo médio (R$)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

#### Conclusão

Ao analisar o custo médio por destino, é possível comparar a eficiência dos gastos entre diferentes localidades, reduzindo a influência da quantidade de viagens realizadas e destacando os destinos com maior impacto financeiro médio.

### Terceira pergunta de negócio - A viagem de maior duração e seu custo total

Esta análise identifica a viagem com a maior duração registrada e apresenta seu custo total, permitindo observar o impacto financeiro associado ao maior período de deslocamento.

In [ ]:
sql = """
SELECT
    id_viagem,
    destinos,
    duracao_dias,
    valor_total
FROM silver_viagem
ORDER BY duracao_dias DESC
LIMIT 1;
"""

df = consultar(sql)

df

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(df["duracao_dias"], df["valor_total"], s=150)
plt.title("Viagem de maior duração")
plt.xlabel("Duração (dias)")
plt.ylabel("Valor Total (R$)")
plt.tight_layout()
plt.show()

#### Conclusão

A análise identificou a viagem com a maior duração registrada e seu respectivo custo total. Esse resultado permite avaliar se períodos mais longos de deslocamento estão associados a maiores despesas, fornecendo informações que podem apoiar o planejamento e o acompanhamento dos gastos com viagens.

### Quarta pergunta de negócio - Qual o tipo de pagamento com maior valor médio?

Nesta análise, o objetivo é identificar qual modalidade de pagamento apresenta o maior valor médio entre as viagens registradas. Essa informação auxilia na compreensão de quais tipos de pagamento concentram despesas mais elevadas, contribuindo para análises de custos e planejamento financeiro da administração pública.

In [ ]:
sql = """
SELECT
    tipo_pagamento,
    AVG(valor_total_pagamentos) AS valor_medio
FROM gold_viagens
GROUP BY tipo_pagamento
ORDER BY valor_medio DESC
LIMIT 1;
"""

df = consultar(sql)

df

In [ ]:
plt.figure(figsize=(7,5))
plt.bar(df["tipo_pagamento"], df["valor_medio"])
plt.title("Tipo de pagamento com maior valor médio")
plt.xlabel("Tipo de pagamento")
plt.ylabel("Valor médio (R$)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

#### Conclusão

A análise identificou a viagem com a maior duração registrada e seu respectivo custo total. Esse resultado permite avaliar se períodos mais longos de deslocamento estão associados a maiores despesas, fornecendo informações que podem apoiar o planejamento e o acompanhamento dos gastos com viagens.

### Quinta pergunta de negócio - Qual o meio de transporte mais usado nos trechos?

Nesta análise, busca-se identificar qual meio de transporte é utilizado com maior frequência nos trechos das viagens oficiais. Essa informação auxilia na compreensão dos padrões de deslocamento e pode subsidiar decisões relacionadas ao planejamento logístico e à gestão dos recursos destinados às viagens.

In [ ]:
sql = """
SELECT
    meio_transporte,
    COUNT(*) AS quantidade
FROM gold_viagens
GROUP BY meio_transporte
ORDER BY quantidade DESC;
"""

df = consultar(sql)

df

In [ ]:
plt.figure(figsize=(6,6))
plt.pie(
    df["quantidade"],
    labels=df["meio_transporte"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Meio de transporte mais utilizado")
plt.show()

#### Conclusão

O resultado mostra o meio de transporte mais utilizado nas viagens registradas. Essa informação permite identificar o modal predominante nos deslocamentos oficiais e pode servir de apoio para análises logísticas e planejamento de custos com transporte.

### Sexta pergunta de negócio - Qual UF de destino aparece em mais trechos?

O objetivo desta análise é identificar qual Unidade Federativa (UF) aparece com maior frequência como destino dos trechos registrados. Esse resultado permite visualizar quais estados concentram a maior parte dos deslocamentos realizados pelos órgãos públicos.

In [ ]:
sql = """
SELECT
    destino_uf,
    COUNT(*) AS quantidade
FROM gold_viagens
GROUP BY destino_uf
ORDER BY quantidade DESC
LIMIT 1;
"""

df = consultar(sql)

df

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(df["destino_uf"], df["quantidade"])
plt.title("UF de destino mais frequente")
plt.xlabel("UF")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.show()

#### Conclusão

A análise identifica a Unidade Federativa que mais aparece como destino dos trechos das viagens oficiais. Esse resultado evidencia a concentração geográfica dos deslocamentos e pode contribuir para estudos sobre a distribuição das viagens realizadas pela administração pública.

### Sétima pergunta de negócio - Qual órgão pagou mais no total?

Nesta análise, busca-se identificar qual órgão pagador realizou o maior volume de pagamentos relacionados às viagens oficiais. Essa informação permite visualizar quais órgãos concentram os maiores gastos com deslocamentos e pode auxiliar no acompanhamento da execução orçamentária.

In [ ]:
sql = """
SELECT
    nome_orgao_pagador,
    SUM(valor_total_pagamentos) AS valor_total
FROM gold_viagens
GROUP BY nome_orgao_pagador
ORDER BY valor_total DESC
LIMIT 1;
"""

df = consultar(sql)

df

In [ ]:
plt.figure(figsize=(10,5))
plt.barh(df["nome_orgao_pagador"], df["valor_total"])
plt.title("Órgão pagador com maior valor total")
plt.xlabel("Valor Total (R$)")
plt.ylabel("Órgão")
plt.tight_layout()
plt.show()

#### Conclusão

A análise mostra qual órgão pagador concentrou o maior volume de recursos destinados ao pagamento das viagens oficiais. Esse resultado pode auxiliar na identificação dos órgãos com maior participação nas despesas de deslocamento e servir de apoio para análises de gestão e controle dos gastos públicos.